<a href="https://colab.research.google.com/github/othmane71-art/ml-intrusion-detection-attack-correlation/blob/main/ML_Powered_Intrusion_Detection_ATTACK_Correlation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Étape 1 : Monter Drive et vérifier les fichiers

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = "/content/drive/MyDrive/AI-Cyber/NSL-KDD-dataset"
print(os.listdir(DATA_DIR))

Mounted at /content/drive
['KDDTest.csv', 'KDDTrain.csv']


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

In [3]:
train = pd.read_csv(f"{DATA_DIR}/KDDTrain.csv")
test = pd.read_csv(f"{DATA_DIR}/KDDTest.csv")
print(train.shape, test.shape)
print(train.columns.tolist())
print(train.iloc[:3, -4:])
print(test.iloc[:3, -4:])

(125973, 43) (22544, 43)
['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack', 'level']
   dst_host_rerror_rate  dst_host_srv_rerror_rate   attack  level
0                  0.05                       0.0   normal     20
1                  0.00                       0.0   normal     15
2                  0.00      

# Étape 2 : Explorer les étiquettes

In [4]:
print(train["attack"].nunique(), "types dans train |", test["attack"].nunique(), "types dans test")
print(train["attack"].value_counts().to_string())

23 types dans train | 38 types dans test
attack
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2


In [5]:
only_test = sorted(set(test["attack"]) - set(train["attack"]))
only_train = sorted(set(train["attack"]) - set(test["attack"]))
print("Absentes du train :", only_test)
print("Absentes du test :", only_train)
print("Lignes de test concernées :", test["attack"].isin(only_test).sum(), "sur", len(test))
print("Valeurs manquantes :", train.isna().sum().sum(), test.isna().sum().sum())
print("Doublons :", train.duplicated().sum(), test.duplicated().sum())
print(train.select_dtypes(include="object").columns.tolist())

Absentes du train : ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']
Absentes du test : ['spy', 'warezclient']
Lignes de test concernées : 3750 sur 22544
Valeurs manquantes : 0 0
Doublons : 0 0
['protocol_type', 'service', 'flag', 'attack']


# Étape 3 : Mapping familles et MITRE ATT&CK

In [6]:
fam = {"normal": "normal"}
for a in "back land neptune pod smurf teardrop apache2 udpstorm processtable worm mailbomb".split(): fam[a] = "DoS"
for a in "satan ipsweep nmap portsweep mscan saint".split(): fam[a] = "Probe"
for a in ("guess_passwd ftp_write imap phf multihop warezmaster warezclient spy xlock xsnoop "
          "snmpguess snmpgetattack httptunnel sendmail named").split(): fam[a] = "R2L"
for a in "buffer_overflow loadmodule rootkit perl sqlattack xterm ps".split(): fam[a] = "U2R"

mitre = {
 "neptune": ("Impact", "T1499.001"), "processtable": ("Impact", "T1499.001"),
 "smurf": ("Impact", "T1498.002"), "udpstorm": ("Impact", "T1498.001"),
 "back": ("Impact", "T1499.004"), "teardrop": ("Impact", "T1499.004"),
 "pod": ("Impact", "T1499.004"), "land": ("Impact", "T1499.004"),
 "apache2": ("Impact", "T1499.002"), "mailbomb": ("Impact", "T1499.002"),
 "worm": ("Lateral Movement", "T1210"),
 "ipsweep": ("Reconnaissance", "T1595.001"), "portsweep": ("Reconnaissance", "T1595.001"),
 "nmap": ("Reconnaissance", "T1595.001"), "satan": ("Reconnaissance", "T1595.002"),
 "saint": ("Reconnaissance", "T1595.002"), "mscan": ("Reconnaissance", "T1595.002"),
 "guess_passwd": ("Credential Access", "T1110.001"), "snmpguess": ("Credential Access", "T1110.001"),
 "ftp_write": ("Initial Access", "T1190"), "imap": ("Initial Access", "T1190"),
 "phf": ("Initial Access", "T1190"), "sendmail": ("Initial Access", "T1190"),
 "named": ("Initial Access", "T1190"),
 "warezclient": ("Command and Control", "T1105"), "warezmaster": ("Command and Control", "T1105"),
 "multihop": ("Lateral Movement", "T1021"), "snmpgetattack": ("Collection", "T1602.001"),
 "xlock": ("Collection", "T1056"), "xsnoop": ("Collection", "T1056"),
 "httptunnel": ("Command and Control", "T1572"), "spy": ("Collection", "A confirmer"),
 "buffer_overflow": ("Privilege Escalation", "T1068"), "loadmodule": ("Privilege Escalation", "T1068"),
 "perl": ("Privilege Escalation", "T1068"), "ps": ("Privilege Escalation", "T1068"),
 "xterm": ("Privilege Escalation", "T1068"), "sqlattack": ("Privilege Escalation", "T1068"),
 "rootkit": ("Defense Evasion", "T1014"),
}

for d in (train, test):
    d["family"] = d["attack"].map(fam)
    d["tactic"] = d["attack"].map(lambda a: mitre.get(a, ("Benign", "-"))[0])
    d["technique"] = d["attack"].map(lambda a: mitre.get(a, ("Benign", "-"))[1])

print("Non mappés :", train["family"].isna().sum(), test["family"].isna().sum())
print(train["family"].value_counts().to_string())
print(test["family"].value_counts().to_string())
print(train[train.family != "normal"].groupby(["family", "tactic", "technique"]).size().to_string())

Non mappés : 0 0
family
normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
family
normal    9711
DoS       7460
R2L       2885
Probe     2421
U2R         67
family  tactic                technique  
DoS     Impact                T1498.002       2646
                              T1499.001      41214
                              T1499.004       2067
Probe   Reconnaissance        T1595.001       8023
                              T1595.002       3633
R2L     Collection            A confirmer        2
        Command and Control   T1105            910
        Credential Access     T1110.001         53
        Initial Access        T1190             23
        Lateral Movement      T1021              7
U2R     Defense Evasion       T1014             10
        Privilege Escalation  T1068             42


# Étape 4 : Préparer les données


In [7]:
drop_cols = ["attack", "level", "family", "tactic", "technique"]
cat = ["protocol_type", "service", "flag"]
classes = ["normal", "DoS", "Probe", "R2L", "U2R"]

X_train = pd.get_dummies(train.drop(columns=drop_cols), columns=cat, dtype=int)
X_test = pd.get_dummies(test.drop(columns=drop_cols), columns=cat, dtype=int)

print("Colonnes absentes du test :", sorted(set(X_train.columns) - set(X_test.columns)))
print("Colonnes propres au test :", sorted(set(X_test.columns) - set(X_train.columns)))

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_train, y_test = train["family"], test["family"]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train :", X_train_s.shape, "| Test :", X_test_s.shape)
print(X_train.columns.equals(X_test.columns))

Colonnes absentes du test : ['service_aol', 'service_harvest', 'service_http_2784', 'service_http_8001', 'service_red_i', 'service_urh_i']
Colonnes propres au test : []
Train : (125973, 122) | Test : (22544, 122)
True


# # Étape 5 : Entraîner le modèle multiclasse

In [8]:
model_plain = LogisticRegression(max_iter=2000, random_state=42)
model_bal = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)

model_plain.fit(X_train_s, y_train)
model_bal.fit(X_train_s, y_train)

print("Itérations :", model_plain.n_iter_, model_bal.n_iter_)
print("Classes :", list(model_bal.classes_))
print("Coefficients :", model_bal.coef_.shape)

Itérations : [190] [808]
Classes : ['DoS', 'Probe', 'R2L', 'U2R', 'normal']
Coefficients : (5, 122)


In [9]:
coef = pd.DataFrame(model_bal.coef_.T, index=X_train.columns, columns=model_bal.classes_)
for c in classes:
    print(f"\n{c} :")
    print(coef[c].sort_values(ascending=False).head(5).round(2).to_string())


normal :
num_root                    4.32
srv_count                   2.68
is_guest_login              2.62
duration                    2.60
dst_host_srv_serror_rate    2.44

DoS :
hot                     6.63
count                   4.89
dst_host_serror_rate    3.87
srv_count               3.39
wrong_fragment          2.88

Probe :
count                       3.93
dst_host_srv_serror_rate    3.03
is_guest_login              2.46
duration                    2.21
rerror_rate                 2.01

R2L :
num_root                    3.18
is_guest_login              3.16
dst_host_srv_serror_rate    3.14
duration                    3.00
rerror_rate                 2.83

U2R :
num_compromised           7.68
same_srv_rate             6.96
dst_host_same_srv_rate    4.39
diff_srv_rate             3.63
protocol_type_udp         3.36


# Étape 6 : Évaluer par classe


In [10]:
def evaluate(m, name):
    pred = m.predict(X_test_s)
    print(f"===== {name} =====")
    print("Accuracy :", round(accuracy_score(y_test, pred) * 100, 2), "%",
          "| Macro-F1 :", round(f1_score(y_test, pred, average="macro", labels=classes), 3))
    cm = pd.DataFrame(confusion_matrix(y_test, pred, labels=classes),
                      index=[f"vrai_{c}" for c in classes], columns=[f"pred_{c}" for c in classes])
    print(cm)
    print(classification_report(y_test, pred, labels=classes, digits=3, zero_division=0))
    return pred

pred_plain = evaluate(model_plain, "Standard")
pred_bal = evaluate(model_bal, "Pondéré (balanced)")

===== Standard =====
Accuracy : 76.5 % | Macro-F1 : 0.59
             pred_normal  pred_DoS  pred_Probe  pred_R2L  pred_U2R
vrai_normal         9031       367         307         0         6
vrai_DoS            1128      6328           4         0         0
vrai_Probe           448        98        1768       107         0
vrai_R2L            2775         3           5        99         3
vrai_U2R              40         2           0         4        21
              precision    recall  f1-score   support

      normal      0.673     0.930     0.781      9711
         DoS      0.931     0.848     0.888      7460
       Probe      0.848     0.730     0.785      2421
         R2L      0.471     0.034     0.064      2885
         U2R      0.700     0.313     0.433        67

    accuracy                          0.765     22544
   macro avg      0.725     0.571     0.590     22544
weighted avg      0.751     0.765     0.724     22544

===== Pondéré (balanced) =====
Accuracy : 78.37 % | 

In [11]:
seen = set(train["attack"])
res = test.assign(pred=pred_bal, ok=(pred_bal == test["family"].values),
                  nouvelle=~test["attack"].isin(seen))
print(res.groupby("nouvelle")["ok"].agg(["mean", "size"]).round(3))
print(res[res.family != "normal"].groupby("tactic")["ok"].agg(["mean", "size"]).round(3))

           mean   size
nouvelle              
False     0.861  18794
True      0.395   3750
                       mean  size
tactic                           
Collection            0.000   191
Command and Control   0.459  1077
Credential Access     0.019  1562
Defense Evasion       0.385    13
Impact                0.847  7458
Initial Access        0.108    37
Lateral Movement      0.150    20
Privilege Escalation  0.481    54
Reconnaissance        0.786  2421


# Étape 7 : Analyse des erreurs et scénarios


In [12]:
idx_n = list(model_bal.classes_).index("normal")
proba = model_bal.predict_proba(X_test_s)
p_att = 1 - proba[:, idx_n]
y_bin = (y_test != "normal").astype(int).values

for t in [0.2, 0.3, 0.5, 0.7, 0.8]:
    pb = (p_att > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_bin, pb).ravel()
    print(f"seuil {t}: FN={fn} FP={fp} recall={tp/(tp+fn):.3f} precision={tp/(tp+fp):.3f}")

det = res[res.family != "normal"].assign(detecte=lambda d: d["pred"] != "normal")
tab = det.groupby("attack").agg(n=("detecte", "size"), detection=("detecte", "mean"), nouvelle=("nouvelle", "first"))
print(tab.sort_values("n", ascending=False).head(15).round(3))

def scenario(nom):
    pos = np.where(test["attack"].values == nom)[0][0]
    print(nom, "| famille réelle:", test["family"].iloc[pos], "| prédit:", model_bal.predict(X_test_s[[pos]])[0])
    print(dict(zip(model_bal.classes_, proba[pos].round(3))))

for a in ["neptune", "guess_passwd", "snmpguess"]:
    scenario(a)

seuil 0.2: FN=2625 FP=988 recall=0.795 precision=0.912
seuil 0.3: FN=2795 FP=900 recall=0.782 precision=0.918
seuil 0.5: FN=3308 FP=831 recall=0.742 precision=0.920
seuil 0.7: FN=3676 FP=771 recall=0.714 precision=0.922
seuil 0.8: FN=3938 FP=729 recall=0.693 precision=0.924
                  n  detection  nouvelle
attack                                  
neptune        4657      0.995     False
guess_passwd   1231      0.310     False
mscan           996      0.729      True
warezmaster     944      0.523     False
apache2         737      0.879      True
satan           735      0.995     False
processtable    685      0.003      True
smurf           665      0.974     False
back            359      0.978     False
snmpguess       331      0.006      True
saint           319      0.984      True
mailbomb        293      0.014      True
snmpgetattack   178      0.034      True
portsweep       157      0.987     False
ipsweep         141      1.000     False
neptune | famille réelle: Do

# **Cellule 1 : Événements réels du test**


In [13]:
fam_mitre = (train[train.family != "normal"].groupby(["family", "tactic", "technique"])
             .size().reset_index(name="n"))
fam_mitre["share"] = fam_mitre["n"] / fam_mitre.groupby("family")["n"].transform("sum")

def mitre_for(family):
    if family == "normal":
        return "-"
    r = fam_mitre[fam_mitre.family == family].nlargest(1, "n").iloc[0]
    return f"{r.tactic} / {r.technique} ({r.share:.0%})"

playbook = {
 "normal": "Aucune action",
 "DoS": "Limiter le débit, SYN cookies, filtrer la source, vérifier la disponibilité",
 "Probe": "Surveiller et bloquer la source, réduire l'exposition (ports/services), threat intel",
 "R2L": "Contrôler les journaux d'authentification du service, verrouiller le compte, MFA",
 "U2R": "Incident : isoler l'hôte, collecter EDR/auditd, vérifier l'élévation de privilèges"}

def verdict(xs, real_family=None):
    p = model_bal.predict_proba(xs)[0]
    pred = model_bal.classes_[p.argmax()]
    if real_family is None:
        statut = "ALERTE" if pred != "normal" else "Normal"
    elif real_family == "normal":
        statut = "OK (vrai négatif)" if pred == "normal" else "FAUSSE ALERTE"
    elif pred == "normal":
        statut = "MANQUEE"
    else:
        statut = "Détectée (bonne famille)" if pred == real_family else "Détectée (mauvaise famille)"
    return pred, p.max(), statut

rows = []
for a in ["neptune", "smurf", "satan", "ipsweep", "guess_passwd", "warezmaster",
          "snmpguess", "processtable", "buffer_overflow", "rootkit", "normal"]:
    pos = np.where(test["attack"].values == a)[0][0]
    pred, conf, statut = verdict(X_test_s[[pos]], test["family"].iloc[pos])
    rows.append({"attaque": a, "famille_reelle": test["family"].iloc[pos], "predite": pred,
                 "conf": round(conf, 2), "statut": statut,
                 "ATTACK_reel": f"{test['tactic'].iloc[pos]} / {test['technique'].iloc[pos]}",
                 "ATTACK_deduit": mitre_for(pred)})
print(pd.DataFrame(rows).to_string(index=False))

        attaque famille_reelle predite  conf                      statut                   ATTACK_reel                      ATTACK_deduit
        neptune            DoS     DoS  1.00    Détectée (bonne famille)            Impact / T1499.001           Impact / T1499.001 (90%)
          smurf            DoS     DoS  1.00    Détectée (bonne famille)            Impact / T1498.002           Impact / T1499.001 (90%)
          satan          Probe   Probe  1.00    Détectée (bonne famille)    Reconnaissance / T1595.002   Reconnaissance / T1595.001 (69%)
        ipsweep          Probe   Probe  1.00    Détectée (bonne famille)    Reconnaissance / T1595.001   Reconnaissance / T1595.001 (69%)
   guess_passwd            R2L     U2R  1.00 Détectée (mauvaise famille) Credential Access / T1110.001 Privilege Escalation / T1068 (81%)
    warezmaster            R2L  normal  0.51                     MANQUEE   Command and Control / T1105                                  -
      snmpguess            R2L  no

# **Cellule 2 : Événements fabriqués**


In [14]:
base = X_test.iloc[[np.where(test["attack"].values == "normal")[0][0]]].copy()

def build(**chg):
    ev = base.copy()
    for k, v in chg.items():
        ev[k] = v
    return ev

events = {
 "SYN flood": build(count=500, srv_count=500, serror_rate=1, srv_serror_rate=1,
                    dst_host_serror_rate=1, dst_host_srv_serror_rate=1, flag_S0=1, flag_SF=0),
 "Scan de ports": build(count=200, diff_srv_rate=1, same_srv_rate=0, rerror_rate=1,
                        srv_rerror_rate=1, flag_REJ=1, flag_SF=0),
 "Brute force FTP": build(num_failed_logins=5, count=30, srv_count=30),
 "Escalade de privilèges": build(root_shell=1, num_root=5, num_compromised=3, su_attempted=1),
}
out = []
for nom, ev in events.items():
    pred, conf, statut = verdict(scaler.transform(ev))
    out.append({"scénario": nom, "predite": pred, "conf": round(conf, 2), "statut": statut,
                "ATTACK_deduit": mitre_for(pred), "réponse": playbook[pred]})
print(pd.DataFrame(out).to_string(index=False))

              scénario predite  conf statut                     ATTACK_deduit                                                                              réponse
             SYN flood     DoS  1.00 ALERTE          Impact / T1499.001 (90%)          Limiter le débit, SYN cookies, filtrer la source, vérifier la disponibilité
         Scan de ports   Probe  0.56 ALERTE  Reconnaissance / T1595.001 (69%) Surveiller et bloquer la source, réduire l'exposition (ports/services), threat intel
       Brute force FTP     R2L  1.00 ALERTE Command and Control / T1105 (91%)     Contrôler les journaux d'authentification du service, verrouiller le compte, MFA
Escalade de privilèges     R2L  1.00 ALERTE Command and Control / T1105 (91%)     Contrôler les journaux d'authentification du service, verrouiller le compte, MFA


# **Cellule 3 : Triage et candidats ATT&CK**


In [15]:
def triage(p_att):
    return "ALERTE" if p_att > 0.5 else ("A INVESTIGUER" if p_att > 0.2 else "Normal")

def candidates(family, k=3):
    if family == "normal":
        return "-"
    sub = fam_mitre[fam_mitre.family == family].nlargest(k, "n")
    return " | ".join(f"{r.technique} ({r.share:.0%})" for r in sub.itertuples())

rows = []
for a in ["neptune", "smurf", "guess_passwd", "warezmaster", "snmpguess", "processtable", "rootkit", "normal"]:
    pos = np.where(test["attack"].values == a)[0][0]
    p = model_bal.predict_proba(X_test_s[[pos]])[0]
    pa = 1 - p[idx_n]
    pred = model_bal.classes_[p.argmax()]
    fam_pred = pred if pred != "normal" else model_bal.classes_[np.argsort(p)[::-1][1]]
    rows.append({"attaque": a, "P(attaque)": round(pa, 2), "triage": triage(pa),
                 "famille_probable": fam_pred, "candidats_ATTACK": candidates(fam_pred),
                 "ATTACK_reel": test["technique"].iloc[pos]})
print(pd.DataFrame(rows).to_string(index=False))

for f in ["DoS", "Probe", "R2L", "U2R"]:
    print(f, "->", candidates(f, 4))

     attaque  P(attaque)        triage famille_probable                                  candidats_ATTACK ATTACK_reel
     neptune        1.00        ALERTE              DoS T1499.001 (90%) | T1498.002 (6%) | T1499.004 (5%)   T1499.001
       smurf        1.00        ALERTE              DoS T1499.001 (90%) | T1498.002 (6%) | T1499.004 (5%)   T1498.002
guess_passwd        1.00        ALERTE              U2R                         T1068 (81%) | T1014 (19%)   T1110.001
 warezmaster        0.49 A INVESTIGUER              R2L         T1105 (91%) | T1110.001 (5%) | T1190 (2%)       T1105
   snmpguess        0.02        Normal            Probe                 T1595.001 (69%) | T1595.002 (31%)   T1110.001
processtable        0.00        Normal              R2L         T1105 (91%) | T1110.001 (5%) | T1190 (2%)   T1499.001
     rootkit        0.27 A INVESTIGUER              R2L         T1105 (91%) | T1110.001 (5%) | T1190 (2%)       T1014
      normal        0.29 A INVESTIGUER              R2L 

Cellule 1 : Générer les logs

In [16]:
import json
rng = np.random.default_rng(7)
label_cols = ["attack", "level", "family", "tactic", "technique"]
feat_cols = [c for c in test.columns if c not in label_cols]

def pick(attack, n):
    idx = np.where(test["attack"].values == attack)[0]
    return test.iloc[rng.choice(idx, size=min(n, len(idx)), replace=False)].copy()

assets = {"SRV-WEB-01": "10.10.20.5", "SRV-FTP-01": "10.10.20.10", "DC-01": "10.10.10.2",
          "SRV-MAIL-01": "10.10.30.15", "SRV-DB-01": "10.10.20.30"}
t0 = pd.Timestamp("2026-10-28 22:00:00")

plan = [
 ("203.0.113.45", "SRV-WEB-01", ["ipsweep"]*3 + ["portsweep"]*3 + ["satan"]*2, 0),
 ("203.0.113.45", "SRV-FTP-01", ["guess_passwd"]*4, 25),
 ("203.0.113.45", "SRV-FTP-01", ["buffer_overflow"]*2 + ["rootkit"], 40),
 ("198.51.100.23", "SRV-WEB-01", ["neptune"]*6, 30),
 ("192.0.2.77", "SRV-DB-01", ["snmpguess"]*4 + ["processtable"]*2, 15),
 ("192.0.2.99", "SRV-MAIL-01", ["warezmaster"]*2, 50),
]
parts = []
for src, host, attacks, start in plan:
    for k, a in enumerate(attacks):
        r = pick(a, 1)
        r["src_ip"], r["dst_host"], r["dst_ip"] = src, host, assets[host]
        r["timestamp"] = t0 + pd.Timedelta(minutes=start, seconds=int(k * rng.integers(20, 90)))
        parts.append(r)

norm = pick("normal", 40)
norm["src_ip"] = rng.choice(["10.10.40.11", "10.10.40.21", "10.10.40.35", "10.10.50.8"], len(norm))
hosts = rng.choice(list(assets), len(norm))
norm["dst_host"] = hosts
norm["dst_ip"] = [assets[h] for h in hosts]
norm["timestamp"] = [t0 + pd.Timedelta(seconds=int(s)) for s in rng.integers(0, 3600, len(norm))]
parts.append(norm)

logs = pd.concat(parts).sort_values("timestamp").reset_index(drop=True)
logs.insert(0, "log_id", [f"EVT-{i:04d}" for i in range(len(logs))])
meta = ["log_id", "timestamp", "src_ip", "dst_ip", "dst_host"]

logs[meta + feat_cols].to_csv(f"{DATA_DIR}/infra_logs.csv", index=False)
logs[["log_id", "attack", "family", "tactic", "technique"]].to_csv(f"{DATA_DIR}/ground_truth.csv", index=False)
print(len(logs), "événements écrits dans", DATA_DIR)
print(logs[meta + ["protocol_type", "service", "flag", "count"]].head(8).to_string(index=False))

69 événements écrits dans /content/drive/MyDrive/AI-Cyber/NSL-KDD-dataset
  log_id           timestamp       src_ip      dst_ip   dst_host protocol_type  service flag  count
EVT-0000 2026-10-28 22:00:00 203.0.113.45  10.10.20.5 SRV-WEB-01          icmp    eco_i   SF      1
EVT-0001 2026-10-28 22:00:18  10.10.40.11 10.10.20.10 SRV-FTP-01           tcp ftp_data   SF      2
EVT-0002 2026-10-28 22:00:51  10.10.40.11 10.10.20.30  SRV-DB-01           tcp     http   SF     10
EVT-0003 2026-10-28 22:01:22 203.0.113.45  10.10.20.5 SRV-WEB-01          icmp    eco_i   SF      1
EVT-0004 2026-10-28 22:01:45 203.0.113.45  10.10.20.5 SRV-WEB-01           tcp  private  REJ     93
EVT-0005 2026-10-28 22:02:00 203.0.113.45  10.10.20.5 SRV-WEB-01           tcp    other  REJ    414
EVT-0006 2026-10-28 22:02:28 203.0.113.45  10.10.20.5 SRV-WEB-01          icmp    eco_i   SF      1
EVT-0007 2026-10-28 22:02:44 203.0.113.45  10.10.20.5 SRV-WEB-01           tcp http_443  REJ      2


In [17]:
SEV = {"U2R": "critical", "R2L": "high", "DoS": "high", "Probe": "medium"}
ACTION = {"DoS": "Limiter le débit, filtrer la source, vérifier la disponibilité",
          "Probe": "Surveiller et bloquer la source, réduire l'exposition",
          "R2L": "Contrôler les logs d'authentification, verrouiller le compte, MFA",
          "U2R": "Incident : isoler l'hôte, collecter EDR/auditd"}
def triage(p): return "ALERTE" if p > 0.5 else ("A INVESTIGUER" if p > 0.2 else "Normal")
def top_tech(f):
    r = fam_mitre[fam_mitre.family == f].nlargest(1, "n").iloc[0]
    return r.tactic, r.technique
def cands(f, k=2):
    return " | ".join(f"{r.technique} ({r.share:.0%})" for r in fam_mitre[fam_mitre.family == f].nlargest(k, "n").itertuples())

logs = pd.read_csv(f"{DATA_DIR}/infra_logs.csv", parse_dates=["timestamp"])
Xl = pd.get_dummies(logs.drop(columns=meta), columns=cat, dtype=int).reindex(columns=X_train.columns, fill_value=0)
proba = model_bal.predict_proba(scaler.transform(Xl))
cl = list(model_bal.classes_)
i_n = cl.index("normal")
att_idx = [i for i, c in enumerate(cl) if c != "normal"]

logs["p_att"] = 1 - proba[:, i_n]
logs["triage"] = logs["p_att"].map(triage)
logs["ml_family"] = [cl[att_idx[int(np.argmax(proba[i, att_idx]))]] for i in range(len(logs))]

al = logs[logs.triage != "Normal"].copy()
al["tactic"] = al.ml_family.map(lambda f: top_tech(f)[0])
al["technique"] = al.ml_family.map(lambda f: top_tech(f)[1])
al["candidats"] = al.ml_family.map(cands)
al["severity"] = al.ml_family.map(SEV)
al["action"] = al.ml_family.map(ACTION)

print(f"=== {len(logs)} événements analysés, {len(al)} alertes ===")
view = al.assign(p_att=al.p_att.round(2))[["timestamp", "src_ip", "dst_host", "service", "ml_family",
                                           "p_att", "triage", "tactic", "technique", "severity"]]
print(view.to_string(index=False))

rank = {"medium": 1, "high": 2, "critical": 3}
inc = []
for ip, g in al.sort_values("timestamp").groupby("src_ip"):
    if not ((g.triage == "ALERTE").any() or len(g) >= 2):
        continue
    tact = list(dict.fromkeys(g.tactic))
    sev = "critical" if len(tact) >= 3 else max(g.severity, key=rank.get)
    inc.append({"src_ip": ip, "alertes": len(g), "debut": g.timestamp.min().strftime("%H:%M:%S"),
                "fin": g.timestamp.max().strftime("%H:%M:%S"), "cibles": ",".join(sorted(set(g.dst_host))),
                "chaine_ATTACK": " -> ".join(tact), "severite": sev})
print("\n=== INCIDENTS CORRELES PAR IP SOURCE ===")
print(pd.DataFrame(inc).sort_values("alertes", ascending=False).to_string(index=False))

gt = pd.read_csv(f"{DATA_DIR}/ground_truth.csv")
chk = logs[["log_id", "src_ip", "triage"]].merge(gt, on="log_id")
miss = chk[(chk.family != "normal") & (chk.triage == "Normal")]
print("\n=== ANGLES MORTS (validation labo, attaques sans alerte) ===")
print(miss.groupby(["src_ip", "attack", "tactic", "technique"]).size().reset_index(name="n").to_string(index=False))
print("Fausses alertes sur trafic normal :", len(chk[(chk.family == "normal") & (chk.triage != "Normal")]), "sur 40")

with open(f"{DATA_DIR}/siem_alerts.jsonl", "w") as f:
    for r in al.itertuples():
        doc = {"@timestamp": r.timestamp.isoformat(),
               "event": {"kind": "alert", "risk_score": int(round(r.p_att * 100)), "severity": r.severity},
               "source": {"ip": r.src_ip}, "destination": {"ip": r.dst_ip}, "host": {"name": r.dst_host},
               "rule": {"name": f"ML-NIDS {r.ml_family}"},
               "threat": {"framework": "MITRE ATT&CK", "tactic": {"name": r.tactic}, "technique": {"id": r.technique}},
               "ml": {"family": r.ml_family, "p_attack": round(float(r.p_att), 3), "triage": r.triage, "candidates": r.candidats},
               "message": r.action}
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")
print("\nExemple d'alerte JSON :")
print(open(f"{DATA_DIR}/siem_alerts.jsonl").readline())

=== 69 événements analysés, 27 alertes ===
          timestamp        src_ip    dst_host  service ml_family  p_att        triage               tactic technique severity
2026-10-28 22:00:00  203.0.113.45  SRV-WEB-01    eco_i     Probe   1.00        ALERTE       Reconnaissance T1595.001   medium
2026-10-28 22:00:18   10.10.40.11  SRV-FTP-01 ftp_data       R2L   0.24 A INVESTIGUER  Command and Control     T1105     high
2026-10-28 22:01:22  203.0.113.45  SRV-WEB-01    eco_i     Probe   1.00        ALERTE       Reconnaissance T1595.001   medium
2026-10-28 22:01:45  203.0.113.45  SRV-WEB-01  private     Probe   1.00        ALERTE       Reconnaissance T1595.001   medium
2026-10-28 22:02:00  203.0.113.45  SRV-WEB-01    other     Probe   1.00        ALERTE       Reconnaissance T1595.001   medium
2026-10-28 22:02:28  203.0.113.45  SRV-WEB-01    eco_i     Probe   1.00        ALERTE       Reconnaissance T1595.001   medium
2026-10-28 22:02:44  203.0.113.45  SRV-WEB-01 http_443     Probe   1.00    

In [18]:
t = test.assign(pred=model_bal.predict(X_test_s))
n = t[t.family == "normal"]
print("Taux de fausses alertes sur trafic normal :")
print(n.groupby(n.service == "private")["pred"].apply(lambda s: (s != "normal").mean()).round(3))
print("\nPart des connexions service=private par famille :")
print((t.service == "private").groupby(t.family).mean().round(3))

Taux de fausses alertes sur trafic normal :
service
False    0.025
True     0.712
Name: pred, dtype: float64

Part des connexions service=private par famille :
family
DoS       0.390
Probe     0.214
R2L       0.173
U2R       0.000
normal    0.088
Name: service, dtype: float64


# Expérience : modèle sans service


In [19]:
cols_ns = [c for c in X_train.columns if not c.startswith("service_")]
ix = [X_train.columns.get_loc(c) for c in cols_ns]

m_ns = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
m_ns.fit(X_train_s[:, ix], y_train)
pred_ns = m_ns.predict(X_test_s[:, ix])

is_n = (y_test == "normal").values
for name, pr in [("avec service", pred_bal), ("sans service", pred_ns)]:
    print(f"{name:13s} accuracy={accuracy_score(y_test, pr):.3f} "
          f"macroF1={f1_score(y_test, pr, average='macro', labels=classes):.3f} "
          f"fausses_alertes={(pr[is_n] != 'normal').mean():.3f} "
          f"detection_attaques={(pr[~is_n] != 'normal').mean():.3f}")

print(classification_report(y_test, pred_ns, labels=classes, digits=3, zero_division=0))

avec service  accuracy=0.784 macroF1=0.583 fausses_alertes=0.085 detection_attaques=0.737
sans service  accuracy=0.809 macroF1=0.625 fausses_alertes=0.060 detection_attaques=0.757
              precision    recall  f1-score   support

      normal      0.745     0.940     0.831      9711
         DoS      0.969     0.828     0.893      7460
       Probe      0.818     0.830     0.824      2421
         R2L      0.911     0.306     0.458      2885
         U2R      0.068     0.507     0.119        67

    accuracy                          0.809     22544
   macro avg      0.702     0.682     0.625     22544
weighted avg      0.847     0.809     0.801     22544



In [20]:
SEV = {"U2R": "high", "R2L": "high", "DoS": "high", "Probe": "medium"}
ACTION = {"DoS": "Limiter le débit, filtrer la source, vérifier la disponibilité",
          "Probe": "Surveiller et bloquer la source, réduire l'exposition",
          "R2L": "Contrôler les logs d'authentification, verrouiller le compte, MFA",
          "U2R": "A confirmer : collecter EDR/auditd, isoler l'hôte si confirmé"}
def triage(p): return "ALERTE" if p > 0.5 else ("A INVESTIGUER" if p > 0.2 else "Normal")
def top_tech(f):
    r = fam_mitre[fam_mitre.family == f].nlargest(1, "n").iloc[0]; return r.tactic, r.technique
def cands(f, k=2):
    return " | ".join(f"{r.technique} ({r.share:.0%})" for r in fam_mitre[fam_mitre.family == f].nlargest(k, "n").itertuples())

logs = pd.read_csv(f"{DATA_DIR}/infra_logs.csv", parse_dates=["timestamp"])
Xl = pd.get_dummies(logs.drop(columns=meta), columns=cat, dtype=int).reindex(columns=X_train.columns, fill_value=0)
proba = m_ns.predict_proba(scaler.transform(Xl)[:, ix])
cl = list(m_ns.classes_)
i_n = cl.index("normal")
att_idx = [i for i, c in enumerate(cl) if c != "normal"]

logs["p_att"] = 1 - proba[:, i_n]
logs["triage"] = logs["p_att"].map(triage)
logs["ml_family"] = [cl[att_idx[int(np.argmax(proba[i, att_idx]))]] for i in range(len(logs))]

al = logs[logs.triage != "Normal"].copy()
al["tactic"] = al.ml_family.map(lambda f: top_tech(f)[0])
al["technique"] = al.ml_family.map(lambda f: top_tech(f)[1])
al["candidats"] = al.ml_family.map(cands)
al["severity"] = al.ml_family.map(SEV)
al["action"] = al.ml_family.map(ACTION)

gt = pd.read_csv(f"{DATA_DIR}/ground_truth.csv")
chk = logs[["log_id", "src_ip", "triage"]].merge(gt, on="log_id")
vrais_ip = set(chk[chk.family != "normal"].src_ip)

print(f"=== {len(logs)} événements, {len(al)} alertes (vraies : {(chk.family != 'normal')[chk.triage != 'Normal'].sum()}) ===")
print(al.assign(p_att=al.p_att.round(2))[["timestamp", "src_ip", "dst_host", "ml_family", "p_att",
      "triage", "tactic", "technique", "severity"]].to_string(index=False))

rank = {"medium": 1, "high": 2, "critical": 3}
inc = []
for ip, g in al.sort_values("timestamp").groupby("src_ip"):
    interne = ip.startswith("10.")
    n_alerte = int((g.triage == "ALERTE").sum())
    if (interne and n_alerte < 2) or (not interne and n_alerte < 1 and len(g) < 2):
        continue
    tact = list(dict.fromkeys(g.tactic))
    sev = "critical" if len(tact) >= 3 else max(g.severity, key=rank.get)
    inc.append({"src_ip": ip, "origine": "interne" if interne else "externe", "alertes": len(g),
                "debut": g.timestamp.min().strftime("%H:%M:%S"), "fin": g.timestamp.max().strftime("%H:%M:%S"),
                "cibles": ",".join(sorted(set(g.dst_host))), "chaine_ATTACK": " -> ".join(tact),
                "severite": sev, "vrai_incident": ip in vrais_ip})
incidents = pd.DataFrame(inc).sort_values("alertes", ascending=False)
print("\n=== INCIDENTS CORRELES ===")
print(incidents.to_string(index=False))
print("Précision incidents :", f"{incidents.vrai_incident.mean():.0%}", "| vrais incidents trouvés :",
      incidents.vrai_incident.sum(), "sur", len(vrais_ip))

miss = chk[(chk.family != "normal") & (chk.triage == "Normal")]
print("\n=== ANGLES MORTS ===")
print(miss.groupby(["src_ip", "attack", "tactic", "technique"]).size().reset_index(name="n").to_string(index=False))
print("Fausses alertes sur trafic normal :", len(chk[(chk.family == "normal") & (chk.triage != "Normal")]), "sur 40")

with open(f"{DATA_DIR}/siem_alerts_v2.jsonl", "w") as f:
    for r in al.itertuples():
        f.write(json.dumps({"@timestamp": r.timestamp.isoformat(),
            "event": {"kind": "alert", "risk_score": int(round(r.p_att * 100)), "severity": r.severity},
            "source": {"ip": r.src_ip}, "destination": {"ip": r.dst_ip}, "host": {"name": r.dst_host},
            "rule": {"name": f"ML-NIDS {r.ml_family}"},
            "threat": {"framework": "MITRE ATT&CK", "tactic": {"name": r.tactic}, "technique": {"id": r.technique}},
            "ml": {"family": r.ml_family, "p_attack": round(float(r.p_att), 3), "triage": r.triage, "candidates": r.candidats},
            "message": r.action}, ensure_ascii=False) + "\n")

=== 69 événements, 30 alertes (vraies : 21) ===
          timestamp        src_ip    dst_host ml_family  p_att        triage               tactic technique severity
2026-10-28 22:00:00  203.0.113.45  SRV-WEB-01     Probe   0.98        ALERTE       Reconnaissance T1595.001   medium
2026-10-28 22:00:18   10.10.40.11  SRV-FTP-01       R2L   0.20 A INVESTIGUER  Command and Control     T1105     high
2026-10-28 22:01:22  203.0.113.45  SRV-WEB-01     Probe   1.00        ALERTE       Reconnaissance T1595.001   medium
2026-10-28 22:01:45  203.0.113.45  SRV-WEB-01     Probe   1.00        ALERTE       Reconnaissance T1595.001   medium
2026-10-28 22:02:00  203.0.113.45  SRV-WEB-01     Probe   1.00        ALERTE       Reconnaissance T1595.001   medium
2026-10-28 22:02:28  203.0.113.45  SRV-WEB-01     Probe   0.99        ALERTE       Reconnaissance T1595.001   medium
2026-10-28 22:02:44  203.0.113.45  SRV-WEB-01     Probe   0.99        ALERTE       Reconnaissance T1595.001   medium
2026-10-28 22:06

In [21]:
def new_sev(ip):
    g = al[al.src_ip == ip]
    s = g[g.p_att >= 0.8]
    if (s.groupby("tactic").size() >= 2).sum() >= 2:
        return "critical"
    if len(s) == 0:
        return "low"
    return max(s.severity, key=rank.get)

incidents["score_moyen"] = incidents.src_ip.map(lambda ip: round(al[al.src_ip == ip].p_att.mean(), 2))
incidents["severite_v3"] = incidents.src_ip.map(new_sev)
print(incidents[["src_ip", "origine", "alertes", "score_moyen", "chaine_ATTACK",
                 "severite", "severite_v3", "vrai_incident"]].to_string(index=False))

       src_ip origine  alertes  score_moyen                                   chaine_ATTACK severite severite_v3  vrai_incident
 203.0.113.45 externe       13         0.96          Reconnaissance -> Privilege Escalation     high    critical           True
198.51.100.23 externe        6         1.00                                          Impact     high        high           True
  10.10.40.35 interne        4         0.47 Impact -> Reconnaissance -> Command and Control critical         low          False
   192.0.2.99 externe        2         0.87                             Command and Control     high        high           True
